# Data Audit

In [1]:
import pandas as pd
import numpy as np

## Load Dataset

In [2]:
from pathlib import Path

data_path = Path("../data/raw")
csv_files = list(data_path.glob("*.csv"))

datasets = {}
for file in csv_files:
    datasets[file.stem] = pd.read_csv(file)

## Dataset Shape

In [3]:
def audit_datasets(datasets):
    rows = []
    for name, df in datasets.items():
        n_rows, n_cols = df.shape
        n_dup_rows = df.duplicated().sum()
        n_unique_rows = df.drop_duplicates().shape[0]
        total_cells = n_rows * n_cols
        missing_cells = df.isna().sum().sum()
        
        rows.append({
            'file': name,
            'rows': n_rows,
            'cols': n_cols,
            'unique_rows': n_unique_rows,
            'dup_rows': n_dup_rows,
            'dup_%': round(n_dup_rows / n_rows * 100, 2) if n_rows else 0,
            'missing_cells': missing_cells,
            'missing_%': round(missing_cells / total_cells * 100, 2) if total_cells else 0,
            'memory_MB': round(df.memory_usage(deep=True).sum() / 1024**2, 2),
            'dtypes': df.dtypes.astype(str).value_counts().to_dict(),
        })
    
    return pd.DataFrame(rows).sort_values('rows', ascending=False).reset_index(drop=True)

audit_datasets(datasets)

,file,rows,cols,unique_rows,dup_rows,dup_%,missing_cells,missing_%,memory_MB,dtypes
0,olist_geolocation_dataset,1000163,5,738332,261831,26.18,0,0.00,146.09,"{'float64': 2, 'object': 2, 'int64': 1}"
1,olist_order_items_dataset,112650,7,112650,0,0.00,0,0.00,39.43,"{'object': 4, 'float64': 2, 'int64': 1}"
2,olist_order_payments_dataset,103886,5,103886,0,0.00,0,0.00,17.81,"{'object': 2, 'int64': 2, 'float64': 1}"
3,olist_customers_dataset,99441,5,99441,0,0.00,0,0.00,29.62,"{'object': 4, 'int64': 1}"
4,olist_orders_dataset,99441,8,99441,0,0.00,4908,0.62,58.97,{'object': 8}
5,olist_order_reviews_dataset,99224,7,99224,0,0.00,145903,21.01,42.75,"{'object': 6, 'int64': 1}"
6,olist_products_dataset,32951,9,32951,0,0.00,2448,0.83,6.79,"{'float64': 7, 'object': 2}"
7,olist_sellers_dataset,3095,4,3095,0,0.00,0,0.00,0.66,"{'object': 3, 'int64': 1}"
8,product_category_name_translation,71,2,71,0,0.00,0,0.00,0.01,{'object': 2}


## Dataset Columns

In [4]:
col_detail = pd.DataFrame([
    {'file': name, 'position': i, 'column': col, 'dtype': str(df[col].dtype), 'missing value': round(df[col].isnull().mean() * 100, 3)}
    for name, df in datasets.items()
    for i, col in enumerate(df.columns)
])
col_detail

,file,position,column,dtype,missing value
0,olist_geolocation_dataset,0,geolocation_zip_code_prefix,int64,0.000
1,olist_geolocation_dataset,1,geolocation_lat,float64,0.000
2,olist_geolocation_dataset,2,geolocation_lng,float64,0.000
3,olist_geolocation_dataset,3,geolocation_city,object,0.000
4,olist_geolocation_dataset,4,geolocation_state,object,0.000
5,olist_order_payments_dataset,0,order_id,object,0.000
6,olist_order_payments_dataset,1,payment_sequential,int64,0.000
7,olist_order_payments_dataset,2,payment_type,object,0.000
8,olist_order_payments_dataset,3,payment_installments,int64,0.000
9,olist_order_payments_dataset,4,payment_value,float64,0.000


In [42]:
id_columns = {
    "olist_orders_dataset": ["order_id", "customer_id"],
    "olist_customers_dataset": ["customer_id", "customer_unique_id"],
    "olist_order_items_dataset": ["order_id", "order_item_id", "product_id", "seller_id"],
    "olist_order_reviews_dataset": ["review_id", "order_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
    "olist_products_dataset": ["product_id"],
    "olist_sellers_dataset": ["seller_id"],
    "olist_geolocation_dataset": ["geolocation_zip_code_prefix"],
    "product_category_name_translation": ["product_category_name"],
}

def analyze_duplicates(datasets, id_columns):
    rows = []
    for name, df in datasets.items():
        for col in id_columns.get(name, []):
            if col in df.columns:
                rows.append({
                    'file': name,
                    'column': col,
                    'unique': df[col].nunique(),
                    'rows': len(df),
                    'duplicated': df[col].duplicated().sum(),
                    'candidate key': '🟢' if df[col].duplicated().sum() == 0 and df[col].nunique() ==len(df) else '🔴'
                })
    return pd.DataFrame(rows)

composite_key = analyze_duplicates(datasets, id_columns)
composite_key

,file,column,unique,rows,duplicated,candidate key
0,olist_geolocation_dataset,geolocation_zip_code_prefix,19015,1000163,981148,🔴
1,olist_order_payments_dataset,order_id,99440,103886,4446,🔴
2,olist_order_payments_dataset,payment_sequential,29,103886,103857,🔴
3,olist_sellers_dataset,seller_id,3095,3095,0,🟢
4,olist_order_items_dataset,order_id,98666,112650,13984,🔴
5,olist_order_items_dataset,order_item_id,21,112650,112629,🔴
6,olist_order_items_dataset,product_id,32951,112650,79699,🔴
7,olist_order_items_dataset,seller_id,3095,112650,109555,🔴
8,olist_order_reviews_dataset,review_id,98410,99224,814,🔴
9,olist_order_reviews_dataset,order_id,98673,99224,551,🔴


In [43]:
id_composite_columns = {
    "olist_order_items_dataset": ["order_id", "order_item_id"],
    "olist_order_reviews_dataset": ["review_id", "order_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
}

def analyze_duplicates(datasets, id_columns):
    rows = []
    for name, df in datasets.items():
        cols = id_columns.get(name, [])
        cols = [c for c in cols if c in df.columns]
        if not cols:
            continue
        rows.append({
            'file': name,
            'columns': ", ".join(cols),
            'unique': df[cols].drop_duplicates().shape[0],
            'rows': len(df),
            'duplicated': df.duplicated(subset=cols).sum(),
            'nulls': int(df[cols].isna().sum().sum()),
            'candidate key': '🟢' if (df.duplicated(subset=cols).sum() == 0 
                                       and df[cols].isna().sum().sum() == 0) 
                              else '🔴'
        })
    return pd.DataFrame(rows)

composite_key = analyze_duplicates(datasets, id_composite_columns)
composite_key

,file,columns,unique,rows,duplicated,nulls,candidate key
0,olist_order_payments_dataset,"order_id, payment_sequential",103886,103886,0,0,🟢
1,olist_order_items_dataset,"order_id, order_item_id",112650,112650,0,0,🟢
2,olist_order_reviews_dataset,"review_id, order_id",99224,99224,0,0,🟢


| Table                | Candidate Primary Key      | Important Foreign Keys                | Grain                                  |
| -------------------- | -------------------------- | ------------------------------------- | -------------------------------------- |
| customers            | `customer_id`              | —                                     | one row per customer/order identity    |
| orders               | `order_id`                 | `customer_id`                         | one row per order                      |
| order_items          | `order_id + order_item_id` | `order_id`, `product_id`, `seller_id` | one row per product item in an order   |
| products             | `product_id`               | —                                     | one row per product                    |
| sellers              | `seller_id`                | —                                     | one row per seller                     |
| payments             | ?                          | `order_id`                            | one row per payment/installment record |
| reviews              | ?                          | `order_id`                            | one row per review                     |
| geolocation          | ?                          | zip prefix                            | one/multiple rows per zip prefix       |
| category translation | `product_category_name`    | —                                     | one row per category                   |
